**课程契约** · 周次 35, 36, 37, 38, 39 · 预计 110 分钟 · Starter 05, 10, 18 · 默认 CPU/离线。

# 02｜MoE 路由与负载

MoE 的关键不是专家数量本身，而是每个 token 选择谁、是否超容量，以及最忙专家是否拖慢整批。

In [ ]:
import torch

from llm_from_scratch.moe import TopKMoE

torch.manual_seed(9)
moe = TopKMoE(16, 32, num_experts=8, top_k=2, capacity_factor=1.0)
x = torch.randn(4, 12, 16)
output, stats = moe(x)
print("output:", output.shape)
print("capacity:", stats["capacity"].item())
print("dropped:", stats["dropped"].sum().item())
print("balance loss:", round(stats["balance_loss"].item(), 4))

In [ ]:
selected = stats["selected_load"].cpu()
accepted = stats["accepted_load"].cpu()
indices = range(len(selected))
print({"selected": selected.tolist(), "accepted": accepted.tolist()})

## 练习

把 Router 权重全部清零。概率会均匀，但离散 Top-k 负载是否均匀？解释 tie-breaking、balance loss 与 z-loss 各自解决什么。

## 学习目标与数据流

`tokens [B,T,D] → router logits [B·T,E] → top-k assignments → capacity/drop → expert FFN → weighted combine`。MoE 增加总容量，但通信和负载不均可抵消算力收益。

In [ ]:
import torch

from llm_from_scratch.moe import TopKMoE

torch.manual_seed(0)
moe = TopKMoE(8, 16, num_experts=4, top_k=2, capacity_factor=1.0)
x = torch.randn(2, 6, 8)
y, info = moe(x)
assert y.shape == x.shape and info["top_indices"].shape == (12, 2)
print(
    {
        "selected": info["selected_load"].tolist(),
        "accepted": info["accepted_load"].tolist(),
        "dropped": int(info["dropped_assignments"]),
    }
)

## 1. Capacity 与 dropping

教学容量为 `ceil(capacity_factor × tokens × top_k / experts)`。selected load 是路由意图，accepted load 是真正执行的 assignment。Dropless 仍有动态调度或 padding/通信权衡。

In [ ]:
tight = TopKMoE(8, 16, num_experts=4, top_k=2, capacity_factor=0.25)
_, ti = tight(x)
assert int(ti["dropped_assignments"]) > 0
print("capacity=", int(ti["capacity"]), "dropped=", int(ti["dropped_assignments"]))

## 2. Top-1 归一化的 router 梯度陷阱

top-k=1 且选中权重重归一为 1 时，主任务沿混合权重对 router 梯度为零；离散 top-k 索引也不可导。router 需 balance/z-loss 或其他稳定策略。

In [ ]:
top1 = TopKMoE(8, 16, num_experts=4, top_k=1, normalize_topk=True)
out, aux = top1(x)
out.square().mean().backward()
main = 0.0 if top1.router.weight.grad is None else float(top1.router.weight.grad.norm())
top1.zero_grad()
_, aux = top1(x)
(aux["balance_loss"] + 0.01 * aux["z_loss"]).backward()
auxg = float(top1.router.weight.grad.norm())
assert main == 0.0 and auxg > 0
print({"main_grad": main, "aux_grad": auxg})

## 3. 总参数、活跃参数与通信分开报告

共享专家总是激活；upcycling 从 dense FFN 初始化；expert parallel 需 dispatch/all-to-all/combine。参数更多不等于每 token FLOPs 同比例增加。

In [ ]:
from llm_from_scratch.moe import expert_parallel_communication_ledger, moe_parameter_accounting

p = moe_parameter_accounting(4096, 11008, 8, 2, shared_experts=1)
c = expert_parallel_communication_ledger(2048, 4096, 2, world_size=8)
print(
    {
        "total_B": round(p["total_parameters"] / 1e9, 2),
        "active_B": round(p["active_parameters_per_token"] / 1e9, 2),
        "remote_MiB": round(c["remote_bytes_uniform_assumption"] / 2**20, 1),
    }
)

## 练习、互动图与来源

使用 `../../interactive/moe_lab.html`，完成 starter 05、11。来源：[Switch](https://arxiv.org/abs/2101.03961)、[GShard](https://arxiv.org/abs/2006.16668)、[OLMoE](https://arxiv.org/abs/2409.02060)、[DeepSeek-V3](https://arxiv.org/abs/2412.19437)。

## 完成断言

- [ ] 区分 selected/accepted load；[ ] 能算 capacity；[ ] 解释 top-1 router 梯度；[ ] 区分 softmax/sigmoid 与重归一；[ ] 报告参数、dropping 和通信假设。